# Process data

In [ ]:
import json

with open('cont_lang_qwen.json','r') as file:
    dataset = json.load(file)

In [ ]:
from collections import defaultdict

grouped = defaultdict(list)
for sample in dataset:
    grouped[sample["story_id"]].append(sample)

data = []
for story_id in sorted(grouped.keys()):
    d = []
    samples = grouped[story_id]
    for sample in samples:
        d.append({
            "story_id": sample["story_id"],
            "lang": sample['lang'],
            "text": sample['response'][0]
        })
    data.append(d)

In [ ]:
lang_list = []
for d in data[0]:
    lang_list.append(d["lang"])

In [ ]:
language_groups = {
    "Western Europe": [
        'fr', 'de', 'nl', 'da', 'sv', 'no', 'fi', 'is', 'ga', 'lb', 'mt'
    ],
    "North America": [
        'en'
    ],
    "Southern Europe": [
        'es', 'pt', 'it', 'ro', 'ca', 'gl', 'el', 'cy'
    ],
    "Eastern Europe & Central Asia": [
        'pl', 'cs', 'sk', 'sl', 'bg', 'mk', 'sr', 'bs', 'be', 'uk', 'ru', 'lt', 'lv', 'et',
        'az', 'kk', 'ky', 'uz', 'tg', 'mn', 'ka', 'ckb'
    ],
    "South Asia": [
        'hi', 'ur', 'bn', 'pa', 'ta', 'te', 'kn', 'ml', 'gu', 'mr', 'ne', 'sd', 'as', 'or'
    ],
    "Southeast Asia": [
        'th', 'lo', 'km', 'vi', 'my', 'tl', 'ceb', 'id', 'ms', 'jv'
    ],
    "East Asia": [
        'zh', 'ja', 'ko'
    ],
    "Middle East & North Africa": [
        'ar', 'fa', 'ps', 'am', 'he', 'so', 'om', 'ha'
    ],
    "Sub-Saharan Africa": [
        'sw', 'yo', 'ig', 'xh', 'zu', 'sn', 'ny', 'wo', 'ln'
    ],
    "Oceania": [
        'mi'
    ]
}

ex_lang = []
for key in language_groups:
    ex_lang.extend(language_groups[key])

In [ ]:
import requests
import time
from tqdm import tqdm

url = ""
MODEL = "Qwen/Qwen3-Embedding-4B"

In [ ]:
import torch
import pickle


for i in range(5):
    results = []
    for sample in tqdm(data[i]):
        user_input = sample['text']

        payload = {
            "model": MODEL,
            "input": user_input,
            "encoding_format": "float"
        }
        headers = {
            "Authorization": "",
            "Content-Type": "application/json"
        }

        response = requests.request("POST", url, json=payload, headers=headers).json()
        pred = torch.tensor(response['data'][0]['embedding'], dtype=torch.float32)
        results.append(pred)
        
    with open("qwen_story_"+str(i+1)+".pkl", "wb") as f:
        pickle.dump(results, f)
        print("qwen_story_"+str(i+1)+" saved.")

# Comparisons Across Languages

## Plot

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import umap

In [ ]:
def plot_scatter(data_tensor, lang_list, title, color):
    np.random.seed(2)
    torch.manual_seed(2)
    data_np = data_tensor.detach().cpu().numpy()

    pca = PCA(n_components=50)
    pca_result = pca.fit_transform(data_np)
    
    avg_tensor = torch.mean(data_tensor, dim=0)                # shape: (2000,)
    avg_np = avg_tensor.detach().cpu().numpy().reshape(1, -1)  # shape: (1, 2000)
    avg_pca = pca.transform(avg_np)                            # shape: (1, 50)

    umap_model = umap.UMAP(n_components=2, random_state=2, transform_seed=42)
    all_data = np.vstack([pca_result, avg_pca])  # shape: (N+1, 50)
    umap_result = umap_model.fit_transform(all_data)  # shape: (N+1, 2)
    points_2d = umap_result[:-1]
    avg_point_2d = umap_result[-1]

    plt.figure(figsize=(8, 6))
    plt.scatter(points_2d[:, 0], points_2d[:, 1], color=color, s=30, alpha=0.6)

    for i, label in enumerate(lang_list):
        plt.text(points_2d[i, 0] + 0.03, points_2d[i, 1], str(label),
                fontsize=9)

    plt.scatter(avg_point_2d[0], avg_point_2d[1], color='red', s=60) #, marker='*'
    plt.text(avg_point_2d[0] + 0.03, avg_point_2d[1], 'AVG', fontsize=9)

    plt.title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
import pickle
embeddings =[]
for i in range(1,6):
    with open('qwen_story_'+str(i)+'.pkl', 'rb') as f:
        emb = pickle.load(f)
    embeddings.append(torch.stack(emb))
embeddings = torch.stack(embeddings)
avg_embeddings = embeddings.mean(dim=0)
plot_scatter(avg_embeddings, lang_list, "Qwen", "turquoise")

## Plot Lang Fam

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import umap
from matplotlib import colors
import matplotlib.cm as cm

In [ ]:
language_groups = {
    "Western Europe": [
        'fr', 'de', 'nl', 'da', 'sv', 'no', 'fi', 'is', 'ga', 'lb', 'mt'
    ],
    "North America": [
        'en'
    ],
    "Southern Europe": [
        'es', 'pt', 'it', 'ro', 'ca', 'gl', 'el', 'cy', 'tr'
    ],
    "Eastern Europe & Central Asia": [
        'pl', 'cs', 'sk', 'sl', 'bg', 'mk', 'sr', 'bs', 'be', 'uk', 'ru', 'lt', 'lv', 'et',
        'az', 'kk', 'ky', 'uz', 'tg', 'mn', 'ka', 'ckb', 'hy', 'hr', 'hu'
    ],
    "South Asia": [
        'hi', 'ur', 'bn', 'pa', 'ta', 'te', 'kn', 'ml', 'gu', 'mr', 'ne', 'sd', 'as', 'or'
    ],
    "Southeast Asia": [
        'th', 'lo', 'km', 'vi', 'my', 'tl', 'ceb', 'id', 'ms', 'jv'
    ],
    "East Asia": [
        'zh', 'ja', 'ko'
    ],
    "Middle East & North Africa": [
        'ar', 'fa', 'ps', 'am', 'he', 'so', 'om', 'ha'
    ],
    "Southern  Africa": [
        'af', 'sw', 'yo', 'ig', 'xh', 'zu', 'sn', 'ny', 'wo', 'ln'
    ],
    "Oceania": [
        'mi'
    ]
}


In [ ]:
def plot_scatter(data_tensor, lang_list, title, language_groups):
    np.random.seed(7)
    torch.manual_seed(7)
    data_np = data_tensor.detach().cpu().numpy()
    
    
    lang_to_family = {lang: fam for fam, langs in language_groups.items() for lang in langs}

    pca = PCA(n_components=50)
    pca_result = pca.fit_transform(data_np)

    avg_tensor = torch.mean(data_tensor, dim=0)
    avg_np = avg_tensor.detach().cpu().numpy().reshape(1, -1)
    avg_pca = pca.transform(avg_np)

    umap_model = umap.UMAP(n_components=2, random_state=2, transform_seed=42)
    all_data = np.vstack([pca_result, avg_pca])
    umap_result = umap_model.fit_transform(all_data)
    points_2d = umap_result[:-1]
    avg_point_2d = umap_result[-1]

    families = sorted(set(lang_to_family.get(lang, "Unknown") for lang in lang_list))
    cmap = cm.get_cmap('tab20', len(families))
    family_to_color = {fam: cmap(i) for i, fam in enumerate(families)}


    plt.figure(figsize=(6, 7))

    for fam in families:
        indices = [i for i, lang in enumerate(lang_list) if lang_to_family.get(lang) == fam]
        if not indices:
            continue
        plt.scatter(points_2d[indices, 0], points_2d[indices, 1],
                    color=family_to_color[fam], s=30, alpha=0.7, label=fam)

    plt.scatter(avg_point_2d[0], avg_point_2d[1], color='red', s=60, marker='*', label='AVG')

    for i, label in enumerate(lang_list):
        plt.text(points_2d[i, 0] + 0.03, points_2d[i, 1], str(label), fontsize=8)

    plt.title(title)
    plt.tight_layout()
    plt.show()


In [ ]:
import pickle
embeddings =[]
for i in range(1,6):
    with open('qwen_story_'+str(i)+'.pkl', 'rb') as f:
        emb = pickle.load(f)
    embeddings.append(torch.stack(emb))
embeddings = torch.stack(embeddings)
avg_embeddings = embeddings.mean(dim=0)
plot_scatter(avg_embeddings, lang_list, "Qwen", language_groups)